# Second CuPy example: loading a CUDA kernel with RawModule

This second notebook computes the same vector sum $c_i=a_i+b_i$ as the first. **The CUDA source, input data, precision, device arrays, and launch geometry are unchanged.** The new step is to create a CUDA module with `cp.RawModule` and retrieve `vector_add` by name, following an organization similar to PyCUDA's `SourceModule` and `get_function`.

A module can contain several kernels. We retain just the familiar addition kernel here to isolate the change of interface. This is a different way to organize CUDA code, not a new numerical algorithm or a promised speedup.

Run the cells in order in a Google Colab GPU runtime. The notebook is self-contained; it does not require running the first notebook. Results and optional timings appear in cells, without exporting files.

## 1. Prepare Google Colab

Select **Runtime → Change runtime type → GPU**, then connect. The commands below run in the remote Linux environment. `nvidia-smi` checks GPU availability; its CUDA version describes driver support. `nvcc --version` reports the installed toolkit compiler.

This notebook uses CuPy's default NVRTC backend to compile the kernel at runtime, rather than invoking `nvcc`. The toolkit check is useful for selecting a CuPy package if none is installed.

In [1]:
!nvidia-smi
!nvcc --version

Sat Oct 17 08:33:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Use the existing CuPy installation when available. The next cell installs a CUDA-specific binary package **only if CuPy is absent**, selecting CUDA 12 or 13 from the installed toolkit. It does not install PyCUDA or a notebook extension. Do not install multiple CuPy packages in the same environment.

If an existing CuPy installation fails to import, use a fresh Colab GPU runtime before proceeding. For other environments, follow the [CuPy installation guide](https://docs.cupy.dev/en/stable/install.html).

In [2]:
import importlib.util
import re
import subprocess
import sys

if importlib.util.find_spec("cupy") is None:
    toolkit_info = subprocess.check_output(
        ["nvcc", "--version"], text=True
    )
    match = re.search(r"release\s+(\d+)\.", toolkit_info)
    if match is None or int(match.group(1)) not in (12, 13):
        raise RuntimeError("Use a Colab GPU runtime with CUDA 12 or 13, or follow the CuPy installation guide.")
    package = f"cupy-cuda{match.group(1)}x"
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    print("Installed:", package)
else:
    print("Using the existing CuPy installation.")

Using the existing CuPy installation.


## 2. Initialize CuPy

`cp` is the conventional alias for CuPy. We use device 0 throughout this standalone notebook. Unlike the PyCUDA example, there is no `pycuda.autoinit` import. CUDA failures are reported as Python exceptions.

In [3]:
import numpy as np
import cupy as cp

if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("No CUDA GPU is available. Select a GPU runtime in Colab.")
cp.cuda.Device(0).use()
properties = cp.cuda.runtime.getDeviceProperties(0)
device_name = properties["name"]
if isinstance(device_name, bytes):
    device_name = device_name.decode()
print("GPU:", device_name)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("CuPy:", cp.__version__)
print("CUDA runtime version:", cp.cuda.runtime.runtimeGetVersion())

GPU: Tesla T4
Python: 3.13.15
NumPy: 2.1.3
CuPy: 14.0.1
CUDA runtime version: 13000


## 3. Choose the vector length and launch size

As in the PyCUDA example, use $N=100003$ and 256 threads per block. Rounding the grid size upward produces a partially filled final block. The CUDA kernel uses 32-bit integer indices; the checks keep the launch within that range and the device limits.

In [4]:
N = 100003
BLOCKSIZE = 256

def iDivUp(a, b):
    return (a + b - 1) // b

assert 1 <= BLOCKSIZE <= min(properties["maxThreadsPerBlock"], properties["maxThreadsDim"][0])
assert 1 <= N <= np.iinfo(np.int32).max - (BLOCKSIZE - 1)
number_of_blocks = iDivUp(N, BLOCKSIZE)
assert number_of_blocks <= properties["maxGridSize"][0]
block = (BLOCKSIZE, 1, 1)
grid = (number_of_blocks, 1, 1)
print(f"Elements: {N}")
print(f"Blocks: {number_of_blocks}; threads per block: {BLOCKSIZE}")
print(f"Threads with no output element: {number_of_blocks * BLOCKSIZE - N}")

Elements: 100003
Blocks: 391; threads per block: 256
Threads with no output element: 93


## 4. Prepare the NumPy inputs

The deterministic inputs match the previous examples: $a_i=0.25(i\bmod101)$ and $b_i=0.50(i\bmod37)$. Both arrays are contiguous `float32` arrays, matching the kernel's `float` pointers.

In [5]:
indices = np.arange(N, dtype=np.int32)
h_a = np.float32(0.25) * (indices % 101).astype(np.float32)
h_b = np.float32(0.50) * (indices % 37).astype(np.float32)
assert h_a.dtype == h_b.dtype == np.float32
assert h_a.flags.c_contiguous and h_b.flags.c_contiguous
print(f"Bytes per vector: {h_a.nbytes}")

Bytes per vector: 400012


## 5. Create device arrays and transfer the inputs

For these **NumPy inputs**, `cp.asarray` allocates device arrays and copies their contents from host to device. In contrast, applying it to an already compatible CuPy array need not make a copy.

`cp.empty_like` allocates the output without initializing its entries. No input transfer is needed for it because the kernel writes every valid element. The CuPy objects manage device storage; no separate raw allocation calls are needed.

All operations below use the same current stream. Keep the host inputs unchanged while their transfers are pending.

In [6]:
d_a = cp.asarray(h_a)
d_b = cp.asarray(h_b)
d_c = cp.empty_like(d_a)
print("Device array type:", type(d_a).__name__)
print("Shape:", d_a.shape, "dtype:", d_a.dtype, "device:", d_a.device.id)

Device array type: ndarray
Shape: (100003,) dtype: float32 device: 0


## 6. Create the module and retrieve the kernel

`RawModule` receives the CUDA source through its `code` argument. `module.get_function("vector_add")` retrieves the kernel by its unmangled name, preserved by `extern "C"`. Retrieving it triggers compilation/loading when needed; CuPy's cache can avoid recompilation. We use the default NVRTC backend and explicitly retain the same C++14 compiler option as the first example.

The returned object is a `RawKernel`, so the subsequent launch syntax is unchanged. `RawModule` also supports retrieving several kernels from one CUDA source; that organizational benefit becomes useful as programs grow. It does not automatically transfer NumPy arguments to the device.

The analogous PyCUDA sequence creates a `SourceModule` and retrieves the function with `get_function`. Here the device storage remains managed by CuPy arrays. Compilation and retrieval take place outside the optional timing interval.

In [7]:
kernel_source = r"""
extern "C" __global__
void vector_add(const float *a, const float *b, float *c, int n)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}
"""

module = cp.RawModule(
    code=kernel_source, options=("-std=c++14",)
)
vector_add = module.get_function("vector_add")
print("Retrieved kernel type:", type(vector_add).__name__)

Retrieved kernel type: RawKernel


## 7. Launch the kernel

CuPy's calling convention is **grid, block, argument tuple**. The argument tuple follows the CUDA declaration: `(a, b, c, n)`. `np.int32(N)` supplies the required 32-bit integer.

Passing the device arrays gives the kernel access to their existing storage; it does not copy them. The launch is asynchronous with respect to Python. We retrieve the result in the next cell using a blocking copy, which waits for the required work in the same stream.

In [8]:
vector_add(grid, block, (d_a, d_b, d_c, np.int32(N)))

## 8. Retrieve and validate every output

`cp.asnumpy` copies device data into a NumPy array. Its default blocking behavior makes the host result available before returning. No separate host synchronization is needed here.

The CPU reference is computed independently. We compare **every entry**, including the final partial block, and reject nonfinite results.

In [9]:
h_c = cp.asnumpy(d_c)
h_reference = h_a + h_b
if not np.all(np.isfinite(h_c)):
    raise AssertionError("The GPU result contains nonfinite values.")
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
max_abs_error = float(np.max(np.abs(h_c - h_reference)))
print(f"Validation passed for all {N} elements.")
print(f"Maximum absolute error: {max_abs_error:.3e}")

Validation passed for all 100003 elements.
Maximum absolute error: 0.000e+00


## 9. Display the computed sum

The `c[i] (GPU)` column is the actual result copied from the GPU. Display the first 12 and last 3 entries, alongside the NumPy reference.

In [10]:
shown_indices = sorted(set(range(min(12, N))) | set(range(max(0, N - 3), N)))
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'c[i] (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_c[i]:15.6f} {h_reference[i]:12.6f}')

       i         a[i]         b[i]      c[i] (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        0.750000     0.750000
       2     0.500000     1.000000        1.500000     1.500000
       3     0.750000     1.500000        2.250000     2.250000
       4     1.000000     2.000000        3.000000     3.000000
       5     1.250000     2.500000        3.750000     3.750000
       6     1.500000     3.000000        4.500000     4.500000
       7     1.750000     3.500000        5.250000     5.250000
       8     2.000000     4.000000        6.000000     6.000000
       9     2.250000     4.500000        6.750000     6.750000
      10     2.500000     5.000000        7.500000     7.500000
      11     2.750000     5.500000        8.250000     8.250000
  100000     2.500000    13.000000       15.500000    15.500000
  100001     2.750000    13.500000       16.250000    16.250000
  100002     3.000000    14.000000      

## 10. Optional: measure the device interval

Warm-up precedes a batch of repeated launches on the same stream. CUDA events measure the elapsed device interval; synchronization of the stop event makes its timestamp available. Compilation, allocation, transfers, validation, and printing are excluded.

This is the same measurement scope as in the first PyCUDA notebook. Short kernels can expose GPU idle gaps between Python submissions, and repeated use of the arrays can benefit from caches. This is neither end-to-end time nor a CPU/GPU speedup. Compare libraries only on the same GPU and under equivalent conditions.

In [11]:
REPEATS = 100
assert REPEATS > 0
stream = cp.cuda.get_current_stream()
for _ in range(3):
    vector_add(grid, block, (d_a, d_b, d_c, np.int32(N)))
stream.synchronize()

start = cp.cuda.Event()
stop = cp.cuda.Event()
start.record(stream)
for _ in range(REPEATS):
    vector_add(grid, block, (d_a, d_b, d_c, np.int32(N)))
stop.record(stream)
stop.synchronize()
elapsed_ms = cp.cuda.get_elapsed_time(start, stop)
mean_ms = elapsed_ms / REPEATS
print(f"Total device interval for {REPEATS} launches: {elapsed_ms:.6f} ms")
print(f"Mean device interval per launch: {mean_ms:.6f} ms")

Total device interval for 100 launches: 2.408224 ms
Mean device interval per launch: 0.024082 ms


## 11. Release the array references

Run this cell when finished. Deleting the last references releases the arrays' ownership of their storage. CuPy's memory pool may retain freed blocks for reuse, so GPU memory reported by monitoring tools need not immediately decrease. Rerun from the device-array cell before launching again.

In [12]:
cp.cuda.get_current_stream().synchronize()
del d_a, d_b, d_c
print("Device array references released.")

Device array references released.


## What changed from the first CuPy example?

| Step | First example | Second example |
|---|---|---|
| CUDA entry point | Construct `RawKernel` from source and a name | Construct `RawModule` from source, then retrieve a kernel by name |
| Device arrays and transfers | `cp.asarray`, `cp.empty_like`, `cp.asnumpy` | Unchanged |
| Kernel launch | Grid, block, argument tuple | Unchanged |
| Numerical operation and validation | Vector sum checked against NumPy | Unchanged |

For one kernel either interface is sufficient. A module is useful for grouping several CUDA kernels in one source and retrieving each by name. This resembles the organization used with PyCUDA's `SourceModule`; it does not introduce an automatic-transfer wrapper like PyCUDA's `In` and `Out`.

The optional timing measures the same repeated device work as the first notebook. Using `RawModule` instead of constructing `RawKernel` directly does not by itself imply faster execution. Compilation is excluded from both measurements.

**Preparation status:** delivered with empty outputs. Python syntax, equality of the CUDA source and unchanged computation cells, and CPU-side index/input checks were verified during preparation. CUDA compilation and execution were not available in the preparation environment. Run all cells on Colab to obtain actual GPU results and timings.

References: [RawModule](https://docs.cupy.dev/en/stable/reference/generated/cupy.RawModule.html), [RawKernel](https://docs.cupy.dev/en/stable/reference/generated/cupy.RawKernel.html).